In [1]:
import json
import pandas as pd
from pathlib import Path
from datetime import datetime

# Load all JSON files
output_dir = Path('.')
results = {}

for json_file in sorted(output_dir.glob('*.json')):
    with open(json_file) as f:
        data = json.load(f)
        
        # Use filename as ID
        experiment_id = json_file.stem
        
        results[experiment_id] = data

print(f"Loaded {len(results)} experiments:")
for exp_id in results.keys():
    print(f"  - {exp_id}")

Loaded 9 experiments:
  - baseline_30_azure-openai-foundry_gpt-4_1
  - baseline_30_meta-llama_Meta-Llama-3_1-70B-Instruct-Turbo
  - baseline_30_meta-llama_Meta-Llama-3_1-8B-Instruct-Turbo
  - baseline_30_mistralai_Mistral-7B-Instruct-v0_2
  - baseline_30_mistralai_Mistral-Small-24B-Instruct-2501
  - full_30_azure-openai-foundry_gpt-4_1
  - full_30_mistralai_Mistral-Small-24B-Instruct-2501
  - guidelines_30_azure-openai-foundry_gpt-4_1
  - guidelines_30_mistralai_Mistral-Small-24B-Instruct-2501


In [2]:
# Create comparison tables
datasets = list(next(iter(results.values()))['datasets'].keys())

# Table 1: Macro F1 scores across datasets
f1_data = []
for model_name, model_data in results.items():
    row = {'Model': model_name}
    for dataset in datasets:
        if dataset in model_data['datasets']:
            f1 = model_data['datasets'][dataset]['reports']['original']['macro avg']['f1-score']
            row[dataset] = round(f1, 3)
    f1_data.append(row)

df_f1 = pd.DataFrame(f1_data).set_index('Model')
print("\n=== Macro F1 Scores by Model and Dataset ===")
print(df_f1.to_string())

# Table 2: Accuracy across datasets
acc_data = []
for model_name, model_data in results.items():
    row = {'Model': model_name}
    for dataset in datasets:
        if dataset in model_data['datasets']:
            acc = model_data['datasets'][dataset]['reports']['original']['accuracy']
            row[dataset] = round(acc, 3)
    acc_data.append(row)

df_acc = pd.DataFrame(acc_data).set_index('Model')
print("\n=== Accuracy by Model and Dataset ===")
print(df_acc.to_string())

# Summary statistics
print("\n=== Average Performance Across All Datasets ===")
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'Avg Macro F1': [df_f1.loc[model].mean() for model in results.keys()],
    'Avg Accuracy': [df_acc.loc[model].mean() for model in results.keys()]
}).set_index('Model')
print(summary.round(3).to_string())

# Optional: Display styled dataframes if running in Jupyter
display(df_f1.style.highlight_max(axis=0, color='lightgreen').format("{:.3f}"))


=== Macro F1 Scores by Model and Dataset ===
                                                          ABSTRCT  ACQUA    AEC    AFS  ARGUMINSCI  FINARG    IAM     PE  SCIARK  USELEC
Model                                                                                                                                   
baseline_30_azure-openai-foundry_gpt-4_1                    0.729  0.766  0.402  0.653       0.764   0.499  0.667  0.433   0.593   0.722
baseline_30_meta-llama_Meta-Llama-3_1-70B-Instruct-Turbo    0.760  0.764  0.525  0.713       0.612   0.444  0.661  0.700   0.633   0.700
baseline_30_meta-llama_Meta-Llama-3_1-8B-Instruct-Turbo     0.625  0.625  0.403  0.403       0.625   0.498  0.466  0.384   0.499   0.524
baseline_30_mistralai_Mistral-7B-Instruct-v0_2              0.623  0.612  0.467  0.760       0.612   0.403  0.700  0.499   0.531   0.833
baseline_30_mistralai_Mistral-Small-24B-Instruct-2501       0.700  0.766  0.633  0.732       0.475   0.403  0.633  0.729   0.525   0

,ABSTRCT,ACQUA,AEC,AFS,ARGUMINSCI,FINARG,IAM,PE,SCIARK,USELEC
Model,,,,,,,,,,
baseline_30_azure-openai-foundry_gpt-4_1,0.729,0.766,0.402,0.653,0.764,0.499,0.667,0.433,0.593,0.722
baseline_30_meta-llama_Meta-Llama-3_1-70B-Instruct-Turbo,0.760,0.764,0.525,0.713,0.612,0.444,0.661,0.700,0.633,0.700
baseline_30_meta-llama_Meta-Llama-3_1-8B-Instruct-Turbo,0.625,0.625,0.403,0.403,0.625,0.498,0.466,0.384,0.499,0.524
baseline_30_mistralai_Mistral-7B-Instruct-v0_2,0.623,0.612,0.467,0.760,0.612,0.403,0.700,0.499,0.531,0.833
baseline_30_mistralai_Mistral-Small-24B-Instruct-2501,0.700,0.766,0.633,0.732,0.475,0.403,0.633,0.729,0.525,0.729
full_30_azure-openai-foundry_gpt-4_1,0.729,nan,nan,nan,0.733,nan,nan,0.542,nan,0.697
full_30_mistralai_Mistral-Small-24B-Instruct-2501,0.667,nan,nan,nan,0.800,nan,nan,0.525,nan,0.764
guidelines_30_azure-openai-foundry_gpt-4_1,0.729,nan,nan,nan,0.764,nan,nan,0.486,nan,0.732
guidelines_30_mistralai_Mistral-Small-24B-Instruct-2501,0.665,nan,nan,nan,0.569,nan,nan,0.665,nan,0.729


In [3]:
# Model Comparison Table: Baseline Robustness Analysis
print("\n" + "="*80)
print("=== Model Comparison: Robustness to Lexical Shortcuts ===")
print("="*80 + "\n")

# Filter only baseline experiments
baseline_results = {k: v for k, v in results.items() if 'baseline' in k}

# Define model metadata (name, size, internal identifier)
model_metadata = {
    'baseline_30_meta-llama_Meta-Llama-3_1-8B-Instruct-Turbo': {
        'name': 'Llama-8B',
        'size': '8B',
        'size_order': 8
    },
    'baseline_30_meta-llama_Meta-Llama-3_1-70B-Instruct-Turbo': {
        'name': 'Llama-70B',
        'size': '70B',
        'size_order': 70
    },
    'baseline_30_mistralai_Mistral-7B-Instruct-v0_2': {
        'name': 'Mistral-7B',
        'size': '7B',
        'size_order': 7
    },
    'baseline_30_mistralai_Mistral-Small-24B-Instruct-2501': {
        'name': 'Mistral-24B',
        'size': '24B',
        'size_order': 24
    },
    'baseline_30_azure-openai-foundry_gpt-4_1': {
        'name': 'GPT-4.1',
        'size': 'frontier',
        'size_order': 1000  # Put frontier models at the end
    }
}

# Compute metrics for each baseline model
comparison_data = []

for model_id, model_data in baseline_results.items():
    if model_id not in model_metadata:
        continue
        
    meta = model_metadata[model_id]
    
    # Calculate mean F1 on original data
    original_f1_scores = []
    feger_deltas = []
    shuffle_deltas = []
    
    for dataset_name, dataset_data in model_data['datasets'].items():
        reports = dataset_data['reports']
        
        # Original F1
        original_f1 = reports['original']['macro avg']['f1-score']
        original_f1_scores.append(original_f1)
        
        # Feger delta
        if 'feger' in reports:
            feger_f1 = reports['feger']['macro avg']['f1-score']
            feger_deltas.append(feger_f1 - original_f1)
        
        # Shuffle delta
        if 'shuffle' in reports:
            shuffle_f1 = reports['shuffle']['macro avg']['f1-score']
            shuffle_deltas.append(shuffle_f1 - original_f1)
    
    comparison_data.append({
        'Model': meta['name'],
        'Size': meta['size'],
        'size_order': meta['size_order'],
        'Mean Δ_feger': round(sum(feger_deltas) / len(feger_deltas), 3) if feger_deltas else None,
        'Mean Δ_shuffle': round(sum(shuffle_deltas) / len(shuffle_deltas), 3) if shuffle_deltas else None,
        'Mean F1 (original)': round(sum(original_f1_scores) / len(original_f1_scores), 3)
    })

# Sort by size
comparison_data = sorted(comparison_data, key=lambda x: x['size_order'])

# Add encoder baseline from Feger et al. (2024) at the end
comparison_data.append({
    'Model': 'Encoders (Feger et al.)',
    'Size': '110–340M',
    'size_order': 0.34,  # For reference
    'Mean Δ_feger': '≤0.02',
    'Mean Δ_shuffle': 'N/A',
    'Mean F1 (original)': 0.79
})

# Create DataFrame and drop the sorting column
df_comparison = pd.DataFrame(comparison_data).drop(columns=['size_order'])

# Display table
print(df_comparison.to_string(index=False))
print("\nNote: More negative Δ values indicate larger performance drops (less robust to manipulation)")
print("Encoders from Feger et al. (2024) show ≤0.02 drop, indicating high robustness.\n")

# Create styled version with color highlighting for delta columns
# Separate numeric rows from reference row for styling
df_numeric = df_comparison[df_comparison['Model'] != 'Encoders (Feger et al.)'].copy()
df_reference = df_comparison[df_comparison['Model'] == 'Encoders (Feger et al.)'].copy()

# Apply styling to numeric rows only
styled = df_numeric.style.background_gradient(
    subset=['Mean Δ_feger', 'Mean Δ_shuffle'],
    cmap='RdYlGn_r',  # Reversed: negative (green) to positive (red)
    vmin=-0.5,
    vmax=0.1
).format({
    'Mean Δ_feger': '{:.3f}',
    'Mean Δ_shuffle': '{:.3f}',
    'Mean F1 (original)': '{:.3f}'
})

display(styled)
print("\n📊 Color guide: 🟢 Green (negative Δ) = more robust | 🔴 Red (positive/less negative Δ) = less robust")
print(f"\n📌 Reference: {df_reference.to_string(index=False, header=False)}")


=== Model Comparison: Robustness to Lexical Shortcuts ===

                  Model     Size Mean Δ_feger Mean Δ_shuffle  Mean F1 (original)
             Mistral-7B       7B       -0.122         -0.213               0.604
               Llama-8B       8B        0.038         -0.023               0.505
            Mistral-24B      24B       -0.208         -0.274               0.632
              Llama-70B      70B       -0.085         -0.129               0.651
                GPT-4.1 frontier       -0.195         -0.269               0.623
Encoders (Feger et al.) 110–340M        ≤0.02            N/A               0.790

Note: More negative Δ values indicate larger performance drops (less robust to manipulation)
Encoders from Feger et al. (2024) show ≤0.02 drop, indicating high robustness.



,Model,Size,Mean Δ_feger,Mean Δ_shuffle,Mean F1 (original)
0,Mistral-7B,7B,-0.122,-0.213,0.604
1,Llama-8B,8B,0.038,-0.023,0.505
2,Mistral-24B,24B,-0.208,-0.274,0.632
3,Llama-70B,70B,-0.085,-0.129,0.651
4,GPT-4.1,frontier,-0.195,-0.269,0.623



📊 Color guide: 🟢 Green (negative Δ) = more robust | 🔴 Red (positive/less negative Δ) = less robust

📌 Reference: Encoders (Feger et al.) 110–340M ≤0.02 N/A 0.79
